In [0]:
dbutils.widgets.text("target_catalog", "workspace", "Target catalog")
dbutils.widgets.text("target_schema", "bronze_bakehouse", "Target schema")
dbutils.widgets.text("source_table", "samples.bakehouse.sales_customers", "Source table (fully qualified)")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
source_table = dbutils.widgets.get("source_table")
target_table = f"{target_catalog}.{target_schema}.customers"

print(f"Source table: {source_table}")
print(f"Target table: {target_table}")

In [0]:
import logging

logger = logging.getLogger("customer_ingestion")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

In [0]:
source_df = spark.read.table(source_table)

if "customerID" not in source_df.columns:
    raise ValueError(f"Source table {source_table} is missing the expected 'customerID' key column")

source_count = source_df.count()
if source_count == 0:
    raise ValueError(f"Source table {source_table} returned zero rows -- aborting load")

logger.info(f"Read {source_count} rows from {source_table}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:target_catalog || '.' || :target_schema);

CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_catalog || '.' || :target_schema || '.customers') (
  customerID     STRING NOT NULL,
  first_name     STRING,
  last_name      STRING,
  _ingested_at   TIMESTAMP,
  _source_table  STRING,
  CONSTRAINT customers_pk PRIMARY KEY (customerID)
) USING DELTA
COMMENT 'Governed customer dimension, upserted from bakehouse sample data. Owner: data-eng.'

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

staged_df = (
    source_df
    .select("customerID", "first_name", "last_name")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_table", F.lit(source_table))
)

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(staged_df.alias("s"), "t.customerID = s.customerID")
    .whenMatchedUpdate(set={
        "first_name": "s.first_name",
        "last_name": "s.last_name",
        "_ingested_at": "s._ingested_at",
        "_source_table": "s._source_table",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

logger.info(f"Merge complete into {target_table}")

In [0]:
target_df = spark.read.table(target_table)
target_count = target_df.count()
null_keys = target_df.filter(F.col("customerID").isNull()).count()

if null_keys > 0:
    raise ValueError(f"{null_keys} rows in {target_table} have a null customerID -- data quality check failed")

if target_count < source_count:
    raise ValueError(
        f"Row count regression: source had {source_count}, target has {target_count} after merge"
    )

logger.info(f"Data quality checks passed: {target_count} rows in {target_table}")

In [0]:
%sql
select * from workspace.bronze_bakehouse.customers